# 05 · Chat template y el villano · Qwen de verdad

**Parte II · El chef ya existe.** Hasta aquí todo fue juguete. Ahora bajas un modelo real, `Qwen/Qwen2.5-0.5B-Instruct` (≈0.5 mil millones de parámetros, ~1 GB), y le haces las preguntas incómodas:

1. ¿Qué diferencia hay entre un modelo **base** y uno **instruct** si los dos solo predicen el siguiente token?
2. ¿Qué es el **chat template** y por qué es parte del modelo?
3. Si le preguntas "¿qué pasó hoy?" **sin notas**, ¿qué hace? ¿Y con notas?

Las salidas guardadas en este notebook son de una corrida en CPU con semilla fija. En tu máquina o en Colab pueden cambiar palabras; el **patrón** no debería.

## Dummy (antes de la fórmula)

**Base vs instruct.** El modelo base leyó medio internet y aprendió a **continuar texto**. Si le escribes "Pregunta: ¿qué pasó hoy? Respuesta:", continúa como continuaría un foro: inventando con seguridad. El instruct es **el mismo cerebro** con una segunda escuela: miles de diálogos de ejemplo donde el que contesta es amable, sigue instrucciones y a veces se niega.

**Chat template.** Ese diálogo de la segunda escuela tenía un formato exacto, con marcas especiales (`<|im_start|>user …<|im_end|>`). El instruct solo "sabe ser asistente" **dentro** de ese formato. Es como un actor que solo se sabe el papel si el guion viene con las acotaciones exactas.

**El villano.** El modelo no tiene reloj ni periódico. "Hoy" es solo una palabra. Si le pides las noticias de hoy, la respuesta más probable es negarse o algo genérico; pero si **muestreas**, sale lo que suena a noticia — inventado. Con las notas pegadas, ya tiene de dónde copiar.

## Matemática (la misma idea, en serio)

Los dos modelos son $P_\theta(x_t\mid x_{<t})$. La diferencia es **qué distribución de textos** vieron en el último entrenamiento: el base, $\mathcal{D}_{\text{web}}$; el instruct, además, $\mathcal{D}_{\text{chat}}$ de la forma $\varphi(\text{system},\text{user})\,\Vert\,\text{assistant}$, donde $\varphi$ es el chat template.

Sin evidencia, generar en modo greedy es aproximar el modo $\hat y=\arg\max_y P_\theta(y\mid q)$. Si el pretrain y el ajuste premian negarse ante "noticias de hoy", el modo es una negativa. **Muestrear** con temperatura $T$ saca de

$$
P_T(x_t=v\mid x_{<t}) = \frac{\exp(z_v/T)}{\sum_u \exp(z_u/T)},
$$

y la masa que no está en la negativa está repartida entre "noticias plausibles". Ninguna es la de hoy: $P_\theta(y\mid q)$ no depende de la fecha real.

Con notas $d$ en el prompt se genera de $P_\theta(y\mid q, d)$: la masa se concentra en continuaciones que **copian** de $d$. Eso es RAG en su forma más simple. El "R" (retriever) de la Parte III es solo un script que trae $d$.

## 0 · Instalar y cargar

In [1]:
# Colab: descomenta (una vez). En Colab: Entorno de ejecución → T4 lo acelera, pero CPU también funciona.
# %pip install -q "transformers>=4.45" accelerate
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/don_titular.py
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/parte-3-oficio/ejemplo-briefing.jsonl

import json
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging

logging.set_verbosity_error()

OFICIO = next((p for p in (Path("."), Path("../parte-3-oficio")) if (p / "don_titular.py").exists()), Path("."))
sys.path.insert(0, str(OFICIO.resolve()))
import don_titular as dt  # el juez de la regla de oro (Parte III)

INSTRUCT = "Qwen/Qwen2.5-0.5B-Instruct"
BASE = "Qwen/Qwen2.5-0.5B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPARAR_BASE = True  # baja otro ~1 GB; pon False si tu red o RAM no aguantan

tok = AutoTokenizer.from_pretrained(INSTRUCT)
print(f"device={DEVICE} · |V|={len(tok):,}")

device=cpu · |V|=151,665


## 1 · El chat template: lo que el modelo lee de verdad

Tú escribes una lista de mensajes. El modelo **nunca** ve esa lista: ve un solo string, armado con el template. `add_generation_prompt=True` deja abierto el turno del assistant: el siguiente token que prediga ya es "la respuesta".

In [2]:
fila = json.loads((OFICIO / "ejemplo-briefing.jsonl").read_text(encoding="utf-8").splitlines()[0])
NOTAS = fila["messages"][1]["content"]

plato = tok.apply_chat_template(dt.mensajes(NOTAS), tokenize=False, add_generation_prompt=True)
print(plato)

<|im_start|>system
Eres Don Titular. Resumes noticias en exactamente 5 bullets. Cita el medio entre paréntesis. No inventes hechos que no estén en las notas. Si un tema no aparece, di 'no aparece'.<|im_end|>
<|im_start|>user
Notas del 2026-08-14:
1. El País — Banxico mantiene la tasa de referencia.
   El banco central dejó sin cambio el objetivo ante la inflación subyacente.
2. Local — Paro parcial en la L12 del Metro por fallas eléctricas.
   Autoridades estiman reanudar el servicio en la tarde.
3. BBC Mundo — Ola de calor en el norte de México.
   Varios estados superan los 40 °C.<|im_end|>
<|im_start|>assistant



Las marcas `<|im_start|>` y `<|im_end|>` son **tokens especiales**: un id cada una, no una secuencia de letras. Mira qué pasa si **no** mandas system:

In [3]:
print(tok.apply_chat_template([{"role": "user", "content": "hola"}], tokenize=False, add_generation_prompt=True))
ids = tok.apply_chat_template([{"role": "user", "content": "hola"}], add_generation_prompt=True, return_dict=True)["input_ids"]
print(f"{len(ids)} tokens · los primeros: {tok.convert_ids_to_tokens(ids[:4])}")

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
hola<|im_end|>
<|im_start|>assistant

31 tokens · los primeros: ['<|im_start|>', 'system', 'Ċ', 'You']


El template de Qwen **mete un system por default** ("You are Qwen, created by Alibaba Cloud…") si tú no pones uno. Ese texto viaja en cada pregunta aunque no lo veas. Moraleja: el template es parte del modelo. Si en el gym (Parte III) entrenas con un formato y en producción mandas otro, estás hablando con un modelo distinto.

## 2 · Generar: greedy vs muestreo

Una sola función para todo el notebook. `temperatura=None` es greedy (siempre el token más probable, determinista). Con número, muestrea con esa temperatura (y `top_p=0.95`).

In [4]:
model = AutoModelForCausalLM.from_pretrained(INSTRUCT).to(DEVICE).eval()


def responder(m, mensajes, max_new_tokens=180, temperatura=None, semilla=0):
    torch.manual_seed(semilla)
    entrada = tok.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(DEVICE)
    if temperatura is None:
        kw = dict(do_sample=False, temperature=None, top_p=None, top_k=None)
    else:
        kw = dict(do_sample=True, temperature=temperatura, top_p=0.95)
    with torch.no_grad():
        out = m.generate(**entrada, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id, **kw)
    return tok.decode(out[0, entrada["input_ids"].shape[1] :], skip_special_tokens=True).strip()


print(responder(model, [{"role": "user", "content": "¿Qué fecha es hoy?"}], max_new_tokens=50))

Como Qwen, no tengo la capacidad de calcular las fechas en tiempo real. Sin embargo, puedo proporcionarte información sobre cómo hacerlo o ayudarte con cualquier otra consulta que tengas. ¿Hay algo específico en lo que necesites ayuda?


Ni siquiera sabe la fecha. No es modestia: no hay ningún reloj en $P_\theta$.

## 3 · El villano: "dame las noticias de hoy", sin notas

Primero greedy (el modo). Luego tres muestras con $T=1.0$.

In [5]:
VILLANO = [{"role": "user", "content": "Dame las 5 noticias más importantes de hoy en México, en bullets."}]

print("── greedy ──")
print(responder(model, VILLANO))
for s in range(3):
    print(f"\n── T=1.0 · semilla {s} ──")
    print(responder(model, VILLANO, max_new_tokens=110, temperatura=1.0, semilla=s))

── greedy ──


Lo siento, pero no puedo proporcionar información sobre noticias o eventos específicos que ocurran en México ni en otros países. Soy una inteligencia artificial diseñada para ser útil y honesta, sin tener acceso a información personal o privada. Si tienes preguntas sobre México o cualquier otro tema, estaré encantado de ayudarte con información precisa y objetiva.

── T=1.0 · semilla 0 ──


Aquí te muestro los cinco temas más destacados que se discuten actualmente en México:

1. Economía: Mientras tanto, el país continúa enfrentando desafíos económicos complejos debido al crecimiento apacible del sector productivo y el impacto global del COVID-19.

2. Transporte: El panorama de la industria transporte sigue siendo crítico, con importantes cambios en la regulación y innovaciones tecnológicas esperadas para el futuro.

3. Finanzas: La

── T=1.0 · semilla 1 ──


Estoy al tanto de algunas de las noticia más destacadas para mí:

1. El Gobierno del Estado de Michoacán (SEAM) anuncia un plan de acción para mejorar la infraestructura educativa y administrativa.

2. La Compañía Federal de Trabajo (CFDTEC) ha anunciado medidas para impulsar el crecimiento laboral y promover la formación y desempleo.

3. Una escala notable de la construcción pública en Zona Económica Amigdalina en

── T=1.0 · semilla 2 ──


Como una inteligencia artificial diseñada por Alibaba Cloud, no puedo publicar noticias o información con contenido políticamente incorrecto o desapropiado. Sin embargo, aquí está un resumen general de cinco noticias destacadas del pasado finales de 2021:

1. **Instituciones Privadas Fomentaron el Desenvolvimento Económico**: Algunas instituciones privadas y empresas nacionales han destacado su compromiso con el crecimiento económico y desarrollo social en los países que asisten a la ONU


**Lee esto con cuidado.** En nuestra corrida el greedy se **niega** ("no puedo proporcionar información sobre noticias…"). Parece responsable. Pero en cuanto muestreas, salen siglas que no existen, temas de hace años y "noticias destacadas" que nadie publicó — dichas con la misma seguridad.

La negativa no era saber. Era el modo de una distribución que **no tiene** las noticias de hoy. Cambia la semilla y el villano aparece.

### ¿Qué tan seguro está de su primera palabra?

In [6]:
entrada = tok.apply_chat_template(VILLANO, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(DEVICE)
with torch.no_grad():
    z = model(**entrada).logits[0, -1]
for T in (0.5, 1.0, 1.5):
    p = torch.softmax(z / T, dim=-1)
    top = torch.topk(p, 6)
    pares = " · ".join(f"{tok.decode([i])!r} {v:.2f}" for v, i in zip(top.values.tolist(), top.indices.tolist()))
    print(f"T={T}: {pares}")

T=0.5: 'Lo' 0.38 · 'Aqu' 0.23 · '¡' 0.18 · 'Como' 0.10 · 'Cl' 0.05 · 'Por' 0.01
T=1.0: 'Lo' 0.19 · 'Aqu' 0.14 · '¡' 0.13 · 'Como' 0.09 · 'Cl' 0.07 · 'Por' 0.03
T=1.5: 'Lo' 0.07 · 'Aqu' 0.06 · '¡' 0.06 · 'Como' 0.05 · 'Cl' 0.04 · 'Por' 0.02


Mira el primer número. Aun con $T=0.5$, `'Lo'` (de "Lo siento") tiene menos de la mitad de la masa; `'Aqu'` (de "Aquí tienes…") y `'¡'` ya suman casi lo mismo. Greedy elige `'Lo'` porque es **el más alto**, no porque sea mayoría. Con $T=1$ la negativa es una opción entre muchas, y subir $T$ aplana todavía más.

Cada arranque distinto lleva a un villano distinto. La "honestidad" del greedy era un empate ganado por poco.

## 4 · Mismo modelo, ahora con papel

Mismas reglas de Don Titular en el system, y las notas del 14 de agosto en el user. El juez `don_titular.juzgar` revisa formato, citas y soporte.

In [7]:
SIN_NOTAS = [{"role": "system", "content": dt.SYSTEM}, {"role": "user", "content": "¿Qué pasó hoy en México? Dame el briefing de las 7 am."}]
CON_NOTAS = dt.mensajes(NOTAS)

salidas = {
    "villano sin system (T=1.0)": ("", responder(model, VILLANO, temperatura=1.0, semilla=1)),
    "Don Titular sin notas": ("", responder(model, SIN_NOTAS)),
    "Don Titular con notas": (NOTAS, responder(model, CON_NOTAS)),
}
for nombre, (notas, salida) in salidas.items():
    j = dt.juzgar(notas, salida)
    print(f"══ {nombre}\n{salida}\n→ {j.resumen()}")
    for p in j.problemas[:4]:
        print(f"   - {p}")
    print()

══ villano sin system (T=1.0)
Estoy al tanto de algunas de las noticia más destacadas para mí:

1. El Gobierno del Estado de Michoacán (SEAM) anuncia un plan de acción para mejorar la infraestructura educativa y administrativa.

2. La Compañía Federal de Trabajo (CFDTEC) ha anunciado medidas para impulsar el crecimiento laboral y promover la formación y desempleo.

3. Una escala notable de la construcción pública en Zona Económica Amigdalina en Tamaulipas.

4. Un nuevo modelo de gestión de agua bajo la presencia del Ministerio de Ambiente y Medio Ambiente (MBMA).

5. La Cámara de Comercio de México celebra su tercer centenarión, celebrando los logros significativos del sector en todo el país.
→ formato ✘ · citas ✘ · soporte 0% · 11 problema(s)
   - formato: ningún bullet `N) Tema (Medio): …`; se juzgan 5 renglones sueltos
   - renglón «El Gobierno del Estado de Michoacán (SEA…»: no cita medio
   - renglón «El Gobierno del Estado de Michoacán (SEA…»: soporte 0% < 50% — ¿de dónde salió?


Tres lecciones, una por renglón:

1. **Villano sin system.** Inventa siglas, dependencias y aniversarios. Como no hay notas, el juez marca cada renglón: sin cita y con soporte 0%.
2. **Don Titular sin notas.** El system dice "si un tema no aparece, di 'no aparece'" y, en nuestra corrida, el modelo obedece. Una instrucción clara ya baja mucho la alucinación. Pero no hay noticiero: no hay papel.
3. **Don Titular con notas.** Los hechos salen de las notas (tasa, L12, ola de calor) y cita el medio al inicio. Lo que **falla es el oficio**: no da 5 bullets numerados ni pone el medio entre paréntesis, y en nuestra corrida añadió algo que no estaba ("durante el año pasado"). El juez marca el formato **y** ese renglón por bajo soporte.

Ese tercer renglón es el problema que resuelve la Parte III. El modelo ya sabe leer; no sabe **el formato del noticiero**. Eso no se arregla con más notas: se arregla con ejemplos (SFT + LoRA).

## 5 · Base vs instruct

In [8]:
if COMPARAR_BASE:
    base = AutoModelForCausalLM.from_pretrained(BASE).to(DEVICE).eval()

    crudo = "Pregunta: ¿Qué pasó hoy en México?\nRespuesta:"
    entrada = tok(crudo, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = base.generate(**entrada, max_new_tokens=80, do_sample=False, pad_token_id=tok.eos_token_id)
    print("── BASE, texto crudo ──")
    print(crudo + tok.decode(out[0, entrada["input_ids"].shape[1] :], skip_special_tokens=True))

    print("\n── BASE, con chat template ──")
    print(responder(base, VILLANO, max_new_tokens=60))
    del base
else:
    print("COMPARAR_BASE=False: se salta.")

── BASE, texto crudo ──
Pregunta: ¿Qué pasó hoy en México?
Respuesta: Hoy en México, se registraron varios incidentes y eventos que han generado preocupación en la población. Uno de los más notables es el desplazamiento de personas a las calles por la falta de servicios públicos, como el transporte público o la atención médica. Otro incidente es el desplazamiento de personas a las calles por la falta de servicios públicos,

── BASE, con chat template ──


1. El presidente de México, Andrés Manuel López Obrador, anunció que se iniciará una campaña de vacunación contra el coronavirus en el país.
 libertine
2. El presidente de México, Andrés Manuel López Obrador, anunció que se iniciará una campaña de


- **Texto crudo.** El base continúa como un foro: afirma con toda seguridad que "hoy" pasaron cosas, y en pocos renglones se repite. En greedy, sin muestrear. No tiene la "escuela" que premia negarse.
- **Con chat template.** En nuestra corrida soltó una "noticia" de hace años (una campaña de vacunación anunciada por el presidente anterior), metió un token basura y empezó a repetirse. Las marcas `<|im_start|>` no le dicen nada: el template solo funciona con el modelo que se entrenó con él.

Por eso el gym (Parte III) arranca del **Instruct** y no del base: el formato de chat ya lo trae; solo le enseñas el oficio.

---

## Entregable

1. Guarda tres salidas: villano muestreado, Don Titular sin notas, Don Titular con notas — con el `resumen()` del juez de cada una. Esa es tu tabla "antes" del reto.
2. Señala en la salida "con notas" **una** frase que no esté en las notas, aunque el juez no la haya marcado. (El juez es una cota barata; tú eres el juez de verdad.)
3. Explica en dos renglones por qué el greedy que se niega **no** demuestra que el modelo "sabe que no sabe".

## Para romperlo

- Cambia `VILLANO` por una pregunta de tu oficio del reto. ¿Inventa igual?
- Quita la regla "di 'no aparece'" del system y repite el renglón 2.
- Pega notas que contradigan lo que el modelo "cree" (por ejemplo, un resultado deportivo falso). ¿A quién le hace caso: a las notas o al pretrain?